## Import Sources

In [ ]:
import pandas as pd
df_imaging=pd.read_csv('cleaned_radio.csv')
df_labs=pd.read_csv('cleaned_lab.csv')
df_truth=pd.read_csv('cleaned_truth.csv')
df_discharge=pd.read_csv('cleaned_discharge_summary.csv')
df_outpatient=pd.read_csv('cleaned_outpatient_summary.csv')
df_endoscope=pd.read_csv('cleaned_endoscope.csv')

## Get Merged Dataframe

In [ ]:
df_patients=(df_truth.copy()[['Random ID']]).dropna(subset=['Random ID'])
df_patients

import pandas as pd

def process_labs(df_labs):
    """Clean, pivot, and flatten lab data."""
    df = df_labs.copy()
    # Parse date safely
    df['timestamp'] = pd.to_datetime(df['Reported Date'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort to preserve chronological order in concatenation
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline for the LLM
    df['Lab_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + " " +
        df['Lab Resulted Order Test Description'].astype(str) + ": " +
        df['Result Value'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Lab_Entry'].apply(' | '.join).reset_index()
def process_imaging(df_imaging):
    df = df_imaging.copy()
    # Remove dayfirst=True to let pandas handle the YYYY-MM-DD format correctly
    df['timestamp'] = pd.to_datetime(df['Performed Date Time'], errors='coerce')
    df = df.dropna(subset=['timestamp'])

    df = df.sort_values(['Random ID', 'timestamp'])
    df['Img_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Text'].astype(str)

    return df.groupby('Random ID', sort=False)['Img_Entry'].apply(' | '.join).reset_index()

def process_discharge(df_discharge):
    """
    Clean and flatten discharge summaries.

    Expected columns:
      - 'Visit Date (YYYYMMDD)' : date in YYYYMMDD format (string or int)
      - 'Full text'             : discharge note content
      - 'Random ID'             : patient identifier
    """
    df = df_discharge.copy()

    # Parse YYYYMMDD robustly (works if the column is str or int)
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort so concatenation is in chronological order
    df = df.sort_values(['Random ID', 'timestamp'])
    # Build entry text
    df['Discharge_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Full text'].astype(str)

    # Group per patient
    return df.groupby('Random ID', sort=False)['Discharge_Entry'].apply(' | '.join).reset_index()
def process_outpatient(df_outpatient):
    """Clean and flatten outpatient clinical notes."""
    df = df_outpatient.copy()
    
    # Filter for relevant clinical notes only
    valid_docs = ['Clinical Note', 'SGH_Consult_ExecSum_TXT']
    df = df[df['Document Item Description'].isin(valid_docs)]
    
    # Parse YYYYMMDD date format
    df['timestamp'] = pd.to_datetime(df['Visit Date (YYYYMMDD)'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # Sort chronologically
    df = df.sort_values(['Random ID', 'timestamp'])

    # Create the string timeline
    df['Outpatient_Entry'] = (
        df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + 
        df['Document Item Description'].astype(str)
    )

    # Group into a single string per patient
    return df.groupby('Random ID', sort=False)['Outpatient_Entry'].apply(' | '.join).reset_index()

def process_endoscopy(df_endoscope):
    """Clean and flatten endoscopy reports."""
    df = df_endoscope.copy()
    
    # 1. Drop exact duplicates, NAs, and short text (< 15 chars)
    df = df.drop_duplicates()
    df = df.dropna(subset=['Summary of Procedure', 'Procedure Start Date'])
    df = df[df['Summary of Procedure'].astype(str).str.len() >= 15]

    # 2. Parse DD/MM/YYYY date safely
    df['timestamp'] = pd.to_datetime(df['Procedure Start Date'], format='%d/%m/%Y', errors='coerce')
    df = df.dropna(subset=['timestamp'])

    # 3. Sort and create entry string
    df = df.sort_values(['Random ID', 'timestamp'])
    df['Endo_Entry'] = df['timestamp'].dt.strftime('%Y-%m-%d') + ": " + df['Summary of Procedure'].astype(str)

    # 4. Group per patient
    return df.groupby('Random ID', sort=False)['Endo_Entry'].apply(' | '.join).reset_index()

def create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope):
    """Merges all sources including endoscopy."""
    labs_processed = process_labs(df_labs)
    imaging_processed = process_imaging(df_imaging)
    discharge_processed = process_discharge(df_discharge)
    outpatient_processed = process_outpatient(df_outpatient)
    endo_processed = process_endoscopy(df_endoscope) # New source

    master = (
        df_patients[['Random ID']]
        .merge(labs_processed, on='Random ID', how='left')
        .merge(imaging_processed, on='Random ID', how='left')
        .merge(discharge_processed, on='Random ID', how='left')
        .merge(outpatient_processed, on='Random ID', how='left')
        .merge(endo_processed, on='Random ID', how='left') # New merge
    )

    # Keep row if at least one narrative text source exists
    text_cols = ['Img_Entry', 'Discharge_Entry', 'Outpatient_Entry', 'Endo_Entry']
    master = master.dropna(subset=text_cols, how='all')

    return master.reset_index(drop=True)

# UPDATED EXECUTION
final_summary = create_master_patient_df(df_patients, df_labs, df_imaging, df_discharge, df_outpatient, df_endoscope)
final_summary

## Main Heuristic and LLM code
Sepsis is included but not used because the results are poor for it (likely missing some unknown sources)

In [1]:
import pandas as pd
import ollama
import asyncio
import json
import re
import os
from pydantic import BaseModel
from tqdm.notebook import tqdm

selected_categories = ["Variceal_bleed", "HE",'Clinical_Ascites','HCC','TIPS',"Portal_Vein_Thrombosis","Spontaneous_Bacterial_Peritonitis"] 

# ==========================================
# 1. CONFIGURATION & TARGETS
# ==========================================
CONCURRENCY_LIMIT = 1
MODEL_NAME = 'qwen3:4b' 
KEYWORD_MAP = {
    "Clinical_Ascites": ["ascites", "ascitic", "shifting dullness","fullness of flank","full flank","distended abdomen","distension of the flank",
                         "distension of abdomen","Abdomen distended","abdo distended", "a distended" "fluid wave","fluid thrill", "paracentesis", "spironolactone"],
    "Radio_Ascites":["ascites", "ascitic", "perihepatic"],
    "Variceal_bleed": ["variceal bleeding", "variceal bleed","bleeding varices", "varices with bleeding", "bleeding esophageal varices","variceal hemorrhage", "active varix", "active variceal", "variceal hemorrhage", 
                        "Active spurting", "Active oozing","spurting","oozing", "fresh blood","altered blood"," acute bleed","haematemesis","red wale", "variceal ligation",
                       "white nipple", "melena"],
    "HE": ["hepatic encephalopathy", "HE" , "portal-systemic encephalopathy", "asterixis", 'flapping tremor', 'tremor flapping',"HE grade", r"(?=.*rifaximin)(?=.*lactulose)", "elevated ammonia"],
    "Spontaneous_Bacterial_Peritonitis": ["spontaneous bacterial peritonitis",r"(?<!secondary )bacterial peritonitis",
            "treated as for SBP", "SBP - Y", "SBP with", "by SBP", "for SBP", "SBP", "PMN", "polymorphonuclear", "neutrophils", 
            r"(?<!secondary )peritonitis","infected peritoneal fluid",  "infected ascitic fluid"], 
    "HCC": ["hepatocellular carcinoma", "HCC", "LI-RADS 5","LI-RADS 4","LIRADS 5","LIRADS 4","hepatoma", "tumor thrombus","liver cancer","tace", "y90"],
    "Portal_Vein_Thrombosis": ["portal vein thrombosis","portal vein thrombus", "PV thrombosis", "MPV thrombosis","LPV thrombosis","RPV thrombosis",
                              "portal thrombosis","portal thrombus", "vein thrombus","vein thrombosis","thrombosis", "thrombus","portal"],
    "TIPS": ["transjugular intrahepatic portosystemic", "TIPS", "TIPSS"],
    "Sepsis":[r'sepsis',r'septic','specticemia','SIRS','CLABSI','CRBSI','BSI','bactermia','lactic acidosis','lactate']
}

CASE_SENSITIVE_KEYWORDS=[
    'HE'
]

class SnippetEvaluation(BaseModel):
    rationale: str
    is_present: int

CATEGORY_PROMPTS = {
    "Spontaneous_Bacterial_Peritonitis": (
        "1. **CORE RULE**: Output 1 if 'SBP' or 'Spontaneous Bacterial Peritonitis' is mentioned in the context of an infection or a complication (e.g., 'BGIT complicated by SBP', 'ppt by SBP', 'SBP undergoing tap'). "
        "Assume the patient HAS SBP if it is mentioned in the diagnosis list or as a current issue, even if the word 'confirmed' is missing.\n"
        "If the Patient has 'SBP prophylaxis' assume to be true and output 1.\n"
        "2. **LABS**: Output 1 if Ascitic WBC × % Neutrophils >= 250, or if 'PMN'/'Polys' are >= 250.\n"
        "3. **NEGATION (The only reasons to output 0)**: \n"
        "   - The mention is clearly SYSTOLIC BLOOD PRESSURE when there are NUMBERS before or after mentioning SBP(e.g., 'SBP 120/80' or 'SBP > 90').\n"
        "   - There is explicit negation: 'No SBP', 'Negative for SBP', 'Not SBP', or 'Rule out SBP' (without follow-up confirmation).\n"
        "   - The mention is clearly FUTURE/PREVENTATIVE: 'Prevention', or 'For SBP KIV'.\n"
        "4. **AMBIGUITY**: If 'SBP' is mentioned and it's not Blood Pressure or explicitly negated, default to 1."
    ),
    "Variceal_bleed": (
        "OVERARCHING RULE: Evaluate each rule independently. If ANY of Rules 1 through 5 are completely met, output 1. Otherwise, output 0.",
        "CRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND):",
        "- 'PR bleed' / 'Per Rectum' = Bleeding from the anus/lower GI tract. This is NEVER a variceal bleed. Output 0."
        "- 'Polypectomy'/ 'Snare' / 'Biopsy base' = Bleeding from a surgical cut by a doctor."
        "RULE 0 (Direct Mention of Variceal Bleed):"
            "Output 1 IF the text directly mentions 'bleeding varices', 'variceal bleed','variceal hemorrhage' is present or was present."
        "RULE 1 (Direct Active Bleed):",
            "Output 1 IF the text explicitly mentions 'active spurting', 'active oozing', 'active bleeding', or 'active hemorrhage' DIRECTLY FROM A VARIX.",
            "CRITICAL EXCLUSION: The blood MUST be coming directly from a varix to output 1"
        "RULE 2 (Blood in Stomach + No Alternate Source):",
            "Output 1 IF there is 'fresh blood', 'altered blood', 'coffee ground material', or 'clots' pooling IN THE STOMACH or GASTRIC FUNDUS, AND the text DOES NOT blame an Ulcer, Gastritis, Esophagitis, or Mallory-Weiss tear.",
        "RULE 3 (High-Risk Signs/Symptoms + No Alternate Source):",
            "Output 1 IF there is 'melena', 'haematemesis', a 'white nipple sign', or a 'red wale sign', and the bleeding source is NOT explicitly identified as an Ulcer, Gastritis, Esophagitis, or Mallory-Weiss tear.",
        "RULE 4 (Historical Bleed):",
            "Output 1 IF the text explicitly states the patient has a 'history of variceal bleed' or a past variceal hemorrhage.",
        "RULE 5 (Therapeutic Procedure):",
            "Output 1 IF 'EVL' or 'Banding' is mentioned AND it is explicitly linked to treating/preventing a bleed. (Fails if labeled ONLY as 'primary prophylaxis' or 'screening').",
        "RULE 6 (Definitive Negatives - Output 0):",
            "Output 0 IF the text states varices have 'no stigmata of recent bleed' and no stomach blood. Output 0 if EVL/Banding is ONLY for 'primary prophylaxis'. Output 0 if all bleeding is explicitly from an ulcer or tear."
    ),
    "TIPS": (
        "CRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND & EXCLUSIONS):\n"
        "- 'TRO' / 'KIV' / 'Planned' / 'Consider' / 'Discussed' / 'Refer to IR' = The TIPS procedure is only a plan or a discussion. The bypass tunnel HAS NOT been built yet. You must wait for definitive proof of the procedure being physically completed to output 1\n"
        "Rule 1 (physical state confirmas existence - output 1):\n"
        "Output 1 IF the text describes the physical state, flow or patency of a TIPS shunt/stent (e.g. 'patent', 'occluded', 'stenosed', 'stunt', describing its physical condition is absolute proof the device is inside the patient.\n)"
    ),
    "HCC": (
        "Only output 1 when there is DIRECT mention of CONFIRMED HCC.\n"
        "IF you see 'resect'/'resection', HCC has occured before, output 1.\n"
        "IF you see 'multifocal HCC', output 1.\n"
        "IF you see 'LI-RADS 5','LI-RADS 4' HCC is HIGHLY LIKELY, and you can immediately output 1.\n"
        "IF you see 'suspicious for' HCC, output 0.\n"
        "For past/history HCC, output 1.\n"
        "CRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND & EXCLUSIONS):\n"
        "- 'HCC surveillance' : IF the text mentions routine surveillance AND the scan is clean/negative (e.g., 'no focal lesions'), output 0. HOWEVER, if the surveillance scan explicitly finds a new tumor, HCC, or Li-RADS 4/5 lesion, you MUST override this and output 1.\n"
        "- 'TRO' / 'KIV' / 'suspected' / 'rule out' = The doctor is guessing. HCC is unconfirmed. Output 0.\n"
        "- 'LI-RADS M' often represents non-HCC malignancies like intrahepatic cholangiocarcinoma or combined HCC-cholangiocarcinoma"
        
    ),
    "Portal_Vein_Thrombosis": (
        "Only output 1 when there is DIRECT mention of a CONFIRMED THROMBUS OR CLOT OR THROMBOSIS in the HEPATIC PORTAL VEIN even if the vein is patent/normal flow. \n"
        "Output 1 when there is mention of some treatment for 'portal vein thrombosis' or 'PVT', meaning the patient does have portal vein thrombosis.\n"
        "CRITICAL REFERENCE DICTIONARY (MEDICAL SHORTHAND & EXCLUSIONS):\n"
        "- 'TRO' / 'KIV' / 'suspected' / 'rule out' = The doctor is guessing. PVT is unconfirmed. Output 0.\n"
        "- 'trend progress' / 'discuss' / 'test for' / 'observation'/ 'compatible with' = PVT is unconfirmed. Output 0.\n"
        "- CONFIRMED 'partial thrombosis' = PVT is confirmed. Output 1.\n"
        "- TRO means TO RULE OUT"
        "- IF 'Observation(s) detected' or 'TRO' is mentioned BEFORE the portal vein thombosis/thrombus, PVT is unconfirmed. Output 0."
        ""
    )
}

# ==========================================
# 2. PYTHON: THE BOUNCER & THE SNIPER
# ==========================================
def the_bouncer_score(sentence, category, keywords):
    """Adds up all applicable score modifiers and returns the total."""
    score = 0
    text_lower = " ".join(sentence.lower().split())
    
    # 1. Affirmative Checks
    if category!="Portal_Vein_Thrombosis":
        affirmative_words = ['yes', 'y', 'present', 'MWA','resection','diagnosed']
        aff_pattern = r"|".join([re.escape(w) for w in affirmative_words])
        for kw in keywords:
            kw_esc = re.escape(kw.lower())
            if re.search(fr"\b{kw_esc}\b(?:\W+\w+){{0,4}}\W+\b({aff_pattern})\b", text_lower):
                score += 500
                break  # Break loop so we don't double-count the same category, but keep checking other criteria

        # 2. Negative Checks
        negative_words = [
            "no", "no previous", "no prev", "suggestive of", "likely", "no evidence of", 
            "no features of", "no objective evidence of", "assess for", "tro", 
            "no history of", "not", "absent", "none", "exclude", "suggest", 
            "denies", "no evidence", "without", "negative for", "ruled out", 
            "unlikely", "possible", "possibly", "discussed", "discuss", 
            "suspect", "consider", "to look for", "nil", "differential", "n/a", 'n'
        ]
        neg_pattern = r"|".join([re.escape(w) for w in negative_words])
        for kw in keywords:
            kw_esc = re.escape(kw.lower())
            if re.search(fr"\b{kw_esc}\b(?:\W+\w+){{0,5}}\W+\b({neg_pattern})\b", text_lower):
                score -= 500
                break
                
    if category=="Spontaneous_Bacterial_Peritonitis":
        num_pattern = r"\b\d+(?:\.\d+)?\b"
        
        for kw in keywords:
            kw_esc = re.escape(kw.lower())
            # Match 1: Keyword followed by a number
            after_pattern = fr"\b{kw_esc}\b(?:\W+\w+){{0,6}}\W+({num_pattern})"
            # Match 2: Number followed by a keyword
            before_pattern = fr"({num_pattern})(?:\W+\w+){{0,6}}\W+\b{kw_esc}\b"
            
            if re.search(after_pattern, text_lower) or re.search(before_pattern, text_lower):
                score -= 500  # Deduct points
                break  # Stop checking this category's keywords to prevent double-counting

   
                
                
    # 3. TIPS specific boosts
    if category == "TIPS":
        tips_boost_keywords = ["stent", "stunt", "patent", "patency", "flow", "functioning"]
        if any(word in text_lower for word in tips_boost_keywords):
            score += 600

    # 4. PVT specific boosts
    if category == "Portal_Vein_Thrombosis":
        if any(word in text_lower for word in ["thrombosis", "thrombus", "clot", "thrombosed", "occluded"]):
            score += 1000
        if any(word in text_lower for word in ["vein", "vin", "venous"]):
            score += 500
        if any(word in text_lower for word in ["portal vein", "pv"]):
            score += 800
        if any(word in text_lower for word in ["splenic", "smv"]):
            score += 900
        if re.search(r"\bportal hypertension\b", text_lower):
            score -= 2000
        if re.search(r"\bnormal\w*\b", text_lower):
            score -= 200
        if re.search(r"\bpatent\b", text_lower):
            score -= 200
        if re.search(r"\bhepatopetal flow\b", text_lower):
            score -= 200

    # 5. HCC specific penalties
    if category == "HCC":
        hcc_ignore = ["surveillance", "screening", "monitoring", "screen", "surveillence",'surveilaince']
        if any(word in text_lower for word in hcc_ignore):
            score -= 900
            
    if category == "HCC":
        hcc_ignore = ["suspicious"]
        if any(word in text_lower for word in hcc_ignore):
            score -= 500

    return score



def is_negated(snippet, keywords, category):
    # Failsafe for empty data
    if not isinstance(snippet, str):
        return False
    lower_snip = snippet.lower()
    # Standard medical negation patterns (made lowercase to match lower_snip)
    if category=='Portal_Vein_Thrombosis':
        neg_pattern = r"(concern for|suspicion|suggestion|no definite evidence of|no|no stigmata of|suspicious of|suspicious for|no hx of|no active|non|no previous|no prev|suggestive of|likely|no evidence of|no previous|no features of|no objective evidence of|assess for|tro|no history of|not|absent|none|exclude|suggest|denies|no evidence|without|negative for|ruled out|unlikely|possible|possibly|discussed|discuss|suspect|consider|to look for|nil|differential)"
    elif category=='HCC':
        neg_pattern = r"(no|suggestive of|rule out|no definite evidence of|no stigmata of|no hx of|no active|non|no previous|no prev|likely|no evidence of|no previous|no features of|no objective evidence of|assess for|tro|no history of|not|absent|none|exclude|suggest|denies|no evidence|without|negative for|ruled out|unlikely|possible|possibly|discussed|discuss|suspect|consider|to look for|nil|differential)"
    else:
        neg_pattern = r"(no|not suggestive of|suggestive of|rule out|no definite evidence of|no stigmata of|no hx of|no active|non|no previous|no prev|likely|no evidence of|no previous|no features of|no objective evidence of|assess for|tro|no history of|not|absent|none|exclude|suggest|denies|no evidence|without|negative for|ruled out|unlikely|possible|possibly|discussed|discuss|suspect|consider|to look for|nil|differential)"
    
    for kw in keywords:
        kw_lower = kw.lower()
        safe_kw = re.escape(kw_lower)
        # 1. CHECK STANDARD NEGATION (For all categories)
        # Using {0,20} characters between the negation and keyword to account for extra words
        if category=="Variceal_bleed":
            if re.search(fr"{neg_pattern}.{{0,10}}\b{safe_kw}\b", lower_snip):
                return True
        elif category=='Portal_Vein_Thrombosis':
            if re.search(fr"{neg_pattern}.{{0,15}}\b{safe_kw}\b", lower_snip):
                return True
        else:
            if re.search(fr"{neg_pattern}.{{0,3}}\b{safe_kw}\b", lower_snip):
                return True
        
        # 3. CHECK THE "OBSERVATION DETECTED" TRAP
        # Using the bulletproof index method so brackets/weird spacing can't break it
        if category == 'Portal_Vein_Thrombosis': # Assuming you wanted to skip PVT here
            obs_index = lower_snip.find("observation")
            det_index = lower_snip.find("detected")
            key_index = lower_snip.find(kw_lower)
            # If all three exist, and 'observation' comes before the keyword
            if obs_index != -1 and det_index != -1 and key_index != -1:
                if obs_index < key_index:
                    return True
    return False
def extract_neighborhoods(text, keywords, category):
    if not isinstance(text, str) or not text.strip():
        return []

    fallback_match = re.search(r'\d{4}-\d{2}-\d{2}', text)
    current_date = fallback_match.group(0) if fallback_match else "No Date Found"
    
    # Splitting sentences (ignoring hyphens)
    raw_sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+|;\s*|\|\s*', text) if s.strip()]
    
    scored_snippets = []
    
    for i, sentence in enumerate(raw_sentences):
        dates_in_sentence = re.findall(r'\d{4}-\d{2}-\d{2}', sentence)
        if dates_in_sentence:
            current_date = dates_in_sentence[-1]

        has_kw = False
        
       
        for kw in keywords:
            # ---> THE DYNAMIC CASE-SENSITIVE CHECK <---
            if kw in CASE_SENSITIVE_KEYWORDS:
                # CASE SENSITIVE: Search original 'sentence'
                if re.search(r'\b' + re.escape(kw) + r'\b', sentence):
                    has_kw = True
                    break
            else:
                # CASE INSENSITIVE: Search 'sentence.lower()'
                if re.search(r'\b' + re.escape(kw.lower()) + r'\b', sentence.lower()):
                    has_kw = True
                    break
                    
        if has_kw and not is_negated(sentence, keywords, category):
            snippet_with_date = f"[Date: {current_date}] {sentence}"
            
            # Call your bouncer function to grade this specific sentence
            snippet_score = the_bouncer_score(sentence, category, keywords)
            
            # Save it as a trio: (Score, Snippet, Date)
            scored_snippets.append((snippet_score, snippet_with_date, current_date))

    # Sort the list based on the snippet_score (highest number first)
    scored_snippets.sort(key=lambda x: (-x[0],x[2]))
    
    # Strip the score back off so we just return the (Snippet, Date) pairs Qwen expects
    final_output = [(snippet, date) for score, snippet, date in scored_snippets]
    
    # Return the TOP 3 HIGHEST SCORING snippets
    return final_output[:3]



# ==========================================
# 3. LLM: THE MICRO-JUDGE
# ==========================================
async def evaluate_snippet(semaphore, snippet, category, patient_id, csv_name): 
    # Only for categories with HCC: Y, HE: N etc. otherwise no need to include, usually under discharge summary
    case_sensitive_cats = ['HE']
    case_insensitive_cats = ['HCC','Hepatocellular carcinoma', "hepatic encephalopathy","Ascites", "PVT", "portal vein thrombosis", "SBP","Variceal Bleed","variceal haemorrhage",'Spontaneous Bacterial Peritonitis']

    affirmative_pattern = r"\s*[-.]?\s*(?:y|yes|present)\b"
    def write_csv(decision, rationale):
        row = pd.DataFrame([{
            "Random ID": patient_id,
            "category": category,
            "decision": decision,
            "rationale": rationale,
            "snippet": snippet
        }])

        row.to_csv(
            csv_name,
            mode="a",                        # append
            header=not os.path.exists(csv_name),
            index=False
        )


    if category in case_sensitive_cats:
        # Strict category label, but flexible affirmative word
        if re.search(rf"{category}:\s*{affirmative_pattern}\b", snippet, re.IGNORECASE):
            # Double check: ensure the category itself is actually uppercase
            if f"{category}:" in snippet:
                print(f"[{category}] ID: {patient_id} - Immediate Match (Strict): 1")
                return 1
    elif category in case_insensitive_cats:
        # Flexible label and flexible affirmative word
        if re.search(rf"{category}:\s*{affirmative_pattern}\b", snippet, re.IGNORECASE):
            print(f"[{category}] ID: {patient_id} - Immediate Match (Relaxed): 1")
            return 1
        
    
    async with semaphore:
        specific_rules = CATEGORY_PROMPTS.get(category, "General clinical auditing rules apply.")
        system_instruction = (
            f"You are to decide whether the patient has {category}.\n"
            f"{specific_rules}\n"
            "Output 1 for confirmed active, historical, secondary/2ndary prophylaxis cases. Output 0 for suspected, or primary prophylaxis.\n"
            "Provide a 1-sentence rationale then the decision clearly. Your rationale must support your decision."
        )
        try:
            
            prompt = f"""
            <clinical_note>
            SNIPPET TO EVALUATE: {snippet}
            <clinical_note>
            """
            
            response = await ollama.AsyncClient().chat(
                model=MODEL_NAME,
                messages=[
                    {'role': 'system', 'content': system_instruction},
                    {'role': 'user', 'content': prompt}
                ],
                format=SnippetEvaluation.model_json_schema(),
                options={'temperature': 0.0, 'num_ctx': 1024, 'seed': 99, "num_gpu": 29}
            )
            
            
            
            current_seed=99
            MAX_RETRIES = 3
            decision, rationale = 0, "Defaulted due to invalid output."

            for attempt in range(MAX_RETRIES):
                current_seed+=42
                try:
                    result = json.loads(response['message']['content'])

                    raw_decision = result.get("is_present", None)
                    rationale = result.get("rationale", "No rationale provided.")

                    # Accept ONLY 0 or 1
                    if isinstance(raw_decision, int) and raw_decision in (0, 1):
                        decision = raw_decision
                        break
                    else:
                        print(f"[Retry {attempt+1}] Invalid decision: {raw_decision}")

                except Exception as e:
                    print(f"[Retry {attempt+1}] Parse error: {str(e)}")

                #re-call model if invalid
                if attempt < MAX_RETRIES - 1:
                    response = await ollama.AsyncClient().chat(
                        model=MODEL_NAME,
                        messages=[
                            {'role': 'system', 'content': system_instruction},
                            {'role': 'user', 'content': prompt}
                        ],
                        format=SnippetEvaluation.model_json_schema(),
                        options={'temperature': 0.0, 'num_ctx': 1024, 'seed': current_seed, 'num_gpu': 29}
                    )

            # final safety fallback
            if decision not in (0, 1):
                print(f"[{category}] ID: {patient_id} - Forced fallback to 0")
                decision = 0
                rationale = "Invalid model output after retries."


            
            
            write_csv(decision, rationale)

            print(f"[{category}] ID: {patient_id}")
            print(f"SNIPPET: {snippet} \n")
            print(f"RATIONALE: {rationale}")
            print(f"DECISION: {decision}\n")

            return decision

        except Exception as e:
            print(f"\n[!!!] FATAL CRASH ON CATEGORY: {category} [!!!]")
            print(f"Snippet: {snippet}")
            print(f"Exact Error: {str(e)}")
            raise

# ==========================================
# 4. THE MAIN PROCESS
# ==========================================
async def process_patient_sniper(semaphore, row, rationale_csv_name):
    patient_id = row.get('Random ID', 'UNKNOWN')
    master_scores = {"Random ID": patient_id}
    for category, keywords in KEYWORD_MAP.items():
        if selected_categories and category not in selected_categories:
            continue
        if category in ["Clinical_Ascites", 'HE', 'Spontaneous_Bacterial_Peritonitis', 'HCC']:
            raw_text = f"DISCHARGE: {row.get('Discharge_Entry', '')}\nOUTPATIENT: {row.get('Outpatient_Entry', '')}"
        elif category in ['Radio_Ascites','Portal_Vein_Thrombosis_presence']: 
            raw_text = f"IMG: {row.get('Img_Entry', '')}"
        elif category=='Variceal_Bleed':
            raw_text = f"DISCHARGE: {row.get('Discharge_Entry', '')}\nOUTPATIENT: {row.get('Outpatient_Entry', '')}\nENDOSCOPY: {row.get('Endo_Entry', '')}"
        else:
            raw_text = f"IMG: {row.get('Img_Entry', '')}\nDISCHARGE: {row.get('Discharge_Entry', '')}\nOUTPATIENT: {row.get('Outpatient_Entry', '')}"
        # Initialize the baseline dictionary entries
        master_scores[f"{category}_presence"] = 0
        master_scores[f"{category}_date"] = "N/A"
        snippets = extract_neighborhoods(raw_text, keywords, category)
        if not snippets:
            continue
        print(f"[{category}] Python found {len(snippets)} mentions. Sending to LLM...")
        # Unpack the Tuple we created in extract_neighborhoods!
        for snippet_text, snippet_date in snippets:
            # Send the date-stamped snippet to Qwen
            is_present = await evaluate_snippet(semaphore, snippet_text, category, patient_id, rationale_csv_name)
            await asyncio.sleep(5)
            if is_present == 1:
                master_scores[f"{category}_presence"] = 1
                master_scores[f"{category}_date"] = snippet_date # Save the extracted date directly to the dict!
                print(f"*** {category} CONFIRMED! Date: {snippet_date}. Skipping remaining snippets for patient {patient_id}***\n")
                break
    return master_scores

# ==========================================
# 5. THE BULK RUNNER
# ==========================================
async def run_sniper_test(df, file_name, rationale_csv_name):
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    tasks = [process_patient_sniper(semaphore, row, rationale_csv_name) for _, row in df.iterrows()]
    results = []
    
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Running Sniper Pipeline"):
        try:
            result = await task
            results.append(result)
            if len(results) > 0 and len(results) % 5 == 0:
                pd.DataFrame(results).to_csv(file_name, index=False)
        except Exception as e:
            print(f"Task failed: {e}")
            
    return pd.DataFrame(results)

Define the categories you want to run in selected_categories, below are some examples:

### Run 1 Event

In [ ]:
selected_categories = ["Variceal_bleed"] 
final_extracted_df= await run_sniper_test(final_summary, "1k VARICEAL 27.5.csv", "1k VARICEAL 27.5 RATIONALE.csv")

### Run all Events

In [ ]:
selected_categories = ["Variceal_bleed", "HE",'Clinical_Ascites','HCC','TIPS',"Portal_Vein_Thrombosis","Spontaneous_Bacterial_Peritonitis"] 
final_extracted_df= await run_sniper_test(final_summary, "1k 27.5.csv", "1k 27.5 RATIONALE.csv")